# 4. PLS loading plot (Figure 1, right panel)

| Output | Corresponding item in the paper |
|---|---|
| `output/figure_1b_loading_plot.png` | Figure 1, loading plot (labelled) |
| `output/figure_1b_loading_plot_rays.png` | same data drawn as rays from the origin |
| `output/pls_loadings.csv` | the loadings of all 166 bits |

**Input:** `output/pls_model.joblib` (run `0_fingerprint_and_pls.ipynb` first),
`data/maccskeys_meaning.csv`.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# Resolve paths relative to the repository root, so that the notebook runs
# both from notebooks/ and from the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
QM = ROOT / "qm" / "qm_nbo_t6311++g"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

SAVE_DPI = 600        # resolution of the saved figure files
plt.rcParams["figure.dpi"] = 100      # on-screen resolution; keeps the notebook fast
plt.rcParams["text.parse_math"] = False   # substructure names may contain "$"

bundle = joblib.load(OUT / "pls_model.joblib")
model = bundle["model"]

# maccskeys_meaning.csv describes bits 1-166; bit 0 is the padding bit and is
# dropped from every per-bit quantity with [1:]
maccs_keys = pd.read_csv(DATA / "maccskeys_meaning.csv")
print(f"{len(maccs_keys)} MACCS bits described, "
      f"{bundle['n_components']} latent variables in the model")

## Loadings of the first two latent variables

`model.x_loadings_` gives the contribution of each MACCS bit to each latent variable.
As with the coefficients, the leading element belongs to the padding bit and is
dropped.

Substructures far from the origin contribute strongly; those in the same direction
behave similarly with respect to the yield.

In [ ]:
maccs_keys["LV1"] = model.x_loadings_[:, 0][1:]
maccs_keys["LV2"] = model.x_loadings_[:, 1][1:]
maccs_keys["LV1_abs"] = maccs_keys["LV1"].abs()
maccs_keys["LV2_abs"] = maccs_keys["LV2"].abs()
maccs_keys = maccs_keys.sort_values("LV1", ascending=False)

maccs_keys.to_csv(OUT / "pls_loadings.csv", index=False)
maccs_keys.head(10)

## Loading plot with substructure labels

In [ ]:
COLOR = "#232A34"

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(maccs_keys["LV1"], maccs_keys["LV2"], color=COLOR, alpha=0.8)
for x, y, name in zip(maccs_keys["LV1"], maccs_keys["LV2"], maccs_keys["subscription"]):
    ax.annotate(name, (x, y), fontsize=9)
ax.set(xlabel="LV1", ylabel="LV2")

fig.savefig(OUT / "figure_1b_loading_plot.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

## The same data drawn as rays from the origin

Drawing a line from the origin to each point makes the direction of every loading
visible even where the labels overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(maccs_keys["LV1"], maccs_keys["LV2"], color=COLOR, alpha=0.8)
for x, y in zip(maccs_keys["LV1"], maccs_keys["LV2"]):
    ax.plot([0, x], [0, y], color=COLOR, linewidth=0.2)
ax.set(xlabel="LV1", ylabel="LV2")

fig.savefig(OUT / "figure_1b_loading_plot_rays.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

Features associated with aromatic rings, six-membered rings and halogen substituents
(F, Cl) load positively onto LV1, whereas oxygen- and nitrogen-containing substructures
load negatively. Read together with the score plot, this is the structural basis for
the observation that weakly coordinating aromatic and chlorinated solvents favour the
Sc(OTf)3-catalyzed reaction.